# ML Assignment 2 - Steel Plate Fault Classification

**Student name:** Anant Tripathi  
**BITS ID:** 2026AJ05781

Trains the 5 models, saves them to `model/` and evaluates them on `test_data.csv`. Run from the repo root.

## 1. Install requirements

In [ ]:
%pip install -q -r requirements.txt

## 2. Load data

In [ ]:
import platform, socket
import joblib, sklearn
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from plate_triage.dataset import TARGET, FAULT_LABELS, load_plates, split_plates
from plate_triage.fit_all import MODEL_DIR, main as fit_everything
from plate_triage.grading import METRIC_NAMES, grade

print('Host:', socket.gethostname(), '| Python', platform.python_version(),
      '| scikit-learn', sklearn.__version__)

plates = load_plates()
train_part, holdout_part = split_plates(plates)
print(f'{len(plates)} plates, {plates.shape[1] - 1} features, '
      f'{plates[TARGET].nunique()} fault classes')
print(f'Train: {len(train_part)} | Holdout: {len(holdout_part)}')
plates[TARGET].value_counts().to_frame('plates')

## 3. Train and save the models (takes a few minutes)

In [ ]:
scorecard = fit_everything()

## 4. Load saved models and evaluate on test_data.csv

In [ ]:
batch = pd.read_csv('test_data.csv')
X, y = batch.drop(columns=TARGET), batch[TARGET]

loaded = {title: joblib.load(MODEL_DIR / f"{entry['slug']}.joblib")
          for title, entry in scorecard['models'].items()}
comparison = pd.DataFrame({
    title: grade(y, model.predict(X), model.predict_proba(X), model.classes_)
    for title, model in loaded.items()
}).T[METRIC_NAMES]
comparison.index.name = 'ML Model Name'
comparison.round(3)

## 5. Train vs CV vs test MCC, and best parameters

In [ ]:
pd.DataFrame({
    title: {'Train MCC': round(e['train_mcc'], 3), 'CV MCC': round(e['cv_mcc'], 3),
            'Holdout MCC': round(e['holdout']['MCC'], 3),
            '95% CI': '{:.3f} to {:.3f}'.format(*e['holdout_mcc_95ci']),
            'Best params': e['best_params']}
    for title, e in scorecard['models'].items()
}).T.style.set_properties(subset=['Best params'], **{'white-space': 'normal'})

## 6. Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (title, model) in zip(axes.flat, loaded.items()):
    ConfusionMatrixDisplay.from_predictions(
        y, model.predict(X), labels=FAULT_LABELS, ax=ax,
        cmap='Blues', colorbar=False, xticks_rotation=45)
    ax.set_title(title)
axes.flat[-1].axis('off')
fig.tight_layout()
plt.show()

## 7. Done

In [ ]:
best = comparison['MCC'].idxmax()
print('=' * 62)
print('Assignment 2 notebook executed successfully.')
print('Student: Anant Tripathi | BITS ID: 2026AJ05781')
print(f'Models saved : {len(list(MODEL_DIR.glob("*.joblib")))} in model/')
print(f'Test plates  : {len(batch)}')
print(f'Best by MCC  : {best} ({comparison.loc[best, "MCC"]:.3f})')
print('=' * 62)